# NDT7 (M-Lab) Data Prep — Vietnam Broadband + Mobile, Province x Quarter

Aggregates `data/ndt7/vn/mlab_vn_clean.parquet` (24.7M raw NDT7 test records, already
ISP-classified and province-joined via per-IP lookup + point-in-polygon) into province x
quarter format, split into Broadband and Mobile/Cellular parts, mirroring
`cambodia_ndt7_prep.ipynb`'s combined structure.

**Rebuilt from the original single-part `vietnam_ndt7_prep.ipynb`** to (a) add a Mobile part
— the raw file has 1.27M cellular-classified rows that the original prep never touched (it
filtered to broadband only) — and (b) process in memory-safe streaming batches rather than
loading all 24.7M rows at once, since this machine has limited free RAM. The tile-binning and
weighted-aggregation formulas are unchanged from the original; Broadband output should
reproduce the original's numbers exactly (mean/count decompose exactly across batches).

Same tile scheme as Ookla's own published tiles (zoom-16 slippy tiles, ~610m) — keeps
`n_tiles`/`is_reliable` comparable across Ookla and NDT7, and across countries:
`total_tests >= 100 & n_tiles >= 5`.

**Outputs:**
- `data/exports/ndt7_vietnam_province_quarterly.csv` — Broadband (same filename as before —
  content should match the prior version)
- `data/exports/ndt7_vietnam_mobile_province_quarterly.csv` — Mobile/Cellular (new)

In [1]:
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import warnings
warnings.filterwarnings('ignore')

RAW_PARQUET = '../../data/ndt7/vn/mlab_vn_clean.parquet'
VN_REF_CSV = '../../data/reference/vietnam_reference.csv'

ZOOM = 16
N_TILES = 2 ** ZOOM
MIN_TILE_TESTS = 3
BATCH_SIZE = 2000000

COLS = ['date', 'mean_throughput_mbps', 'min_rtt', 'latitude', 'longitude',
        'type', 'network_type', 'province']

### 1. Streaming Tile-Binning + Partial Aggregation Helpers

No province-name mapping needed — Vietnam's raw `province` values already match `vietnam_reference.csv` (unlike Cambodia's diacritic mismatch).

In [2]:
PROVINCE_MAP_FN = lambda s: s

def tile_bin_batch(df):
    """Assign zoom-16 tile IDs and quarter labels to a raw batch (no accumulation)."""
    df = df[df['mean_throughput_mbps'] > 0]
    df = df.dropna(subset=['latitude', 'longitude', 'province', 'date'])
    df = df.copy()
    df['province'] = PROVINCE_MAP_FN(df['province'])
    df = df.dropna(subset=['province'])
    if df.empty:
        return df

    df['date'] = pd.to_datetime(df['date'])
    df['year_q'] = (
        df['date'].dt.to_period('Q').astype(str)
        .str.replace(r'(\d{4})Q(\d)', r'\1-Q\2', regex=True)
    )
    df['min_rtt'] = df['min_rtt'].clip(upper=2000)

    lat_rad = np.radians(df['latitude'].clip(-85.05112878, 85.05112878))
    mercator_y = np.log(np.tan(lat_rad) + 1.0 / np.cos(lat_rad))
    df['tile_x'] = ((df['longitude'].astype(float) + 180) / 360 * N_TILES).astype(int).clip(0, N_TILES - 1)
    df['tile_y'] = ((1 - mercator_y / np.pi) / 2 * N_TILES).astype(int).clip(0, N_TILES - 1)
    df['tile_id'] = df['tile_x'].astype(str) + '_' + df['tile_y'].astype(str)
    return df


def partial_tile_agg(df):
    """Per-batch partial sums at (year_q, tile_id, type, network_type) grain — small output,
    safe to accumulate across many batches. sum/count recombine exactly like a single-pass
    groupby would (mean-of-batch-means would NOT be exact; sum/count is)."""
    return df.groupby(['year_q', 'tile_id', 'type', 'network_type']).agg(
        sum_throughput=('mean_throughput_mbps', 'sum'),
        sum_rtt=('min_rtt', 'sum'),
        n=('mean_throughput_mbps', 'count'),
        province=('province', lambda s: s.mode().iat[0]),
    ).reset_index()


def combine_partials(parts):
    """Sum per-batch partials down to one row per (year_q, tile_id, type, network_type)."""
    allp = pd.concat(parts, ignore_index=True)
    combined = allp.groupby(['year_q', 'tile_id', 'type', 'network_type']).agg(
        sum_throughput=('sum_throughput', 'sum'),
        sum_rtt=('sum_rtt', 'sum'),
        n=('n', 'sum'),
        # province is consistent per tile (tiles are ~610m, city-level geolocation almost
        # always unanimous) — take the mode of per-batch modes as a cheap approximation
        province=('province', lambda s: s.mode().iat[0]),
    ).reset_index()
    combined['tile_mean'] = combined['sum_throughput'] / combined['n']
    combined['tile_lat']  = combined['sum_rtt'] / combined['n']
    return combined

### 2. Stream Through Raw Parquet in Batches

In [3]:
pf = pq.ParquetFile(RAW_PARQUET)
print(f"Total rows in file: {pf.metadata.num_rows:,}")

parts = []
rows_seen = 0
for bi, batch in enumerate(pf.iter_batches(columns=COLS, batch_size=BATCH_SIZE)):
    raw_batch = batch.to_pandas()
    rows_seen += len(raw_batch)
    binned = tile_bin_batch(raw_batch)
    if not binned.empty:
        parts.append(partial_tile_agg(binned))
    del raw_batch, binned
    if (bi + 1) % 10 == 0:
        print(f"  processed {rows_seen:,} raw rows so far...")

print(f"Done: {rows_seen:,} raw rows read, {len(parts)} batch-partials to combine")
tile_agg_all = combine_partials(parts)
del parts
tile_agg_all = tile_agg_all[tile_agg_all['n'] >= MIN_TILE_TESTS].copy()
print(f"Tile x quarter x type x network rows after MIN_TILE_TESTS>={MIN_TILE_TESTS} filter: {len(tile_agg_all):,}")
print(tile_agg_all['network_type'].unique())

Total rows in file: 24,667,548


  processed 20,000,000 raw rows so far...


Done: 24,667,548 raw rows read, 13 batch-partials to combine


Tile x quarter x type x network rows after MIN_TILE_TESTS>=3 filter: 7,062
<ArrowStringArray>
['broadband', 'cellular', 'hosting']
Length: 3, dtype: str


### 3. Province-Level Weighted Aggregation (per network type)

In [4]:
def build_province_quarterly(tile_agg_all, network_type, ref):
    tile_agg = tile_agg_all[tile_agg_all['network_type'] == network_type]
    print(f"[{network_type}] tile x quarter x type rows: {len(tile_agg):,}")

    dl = tile_agg[tile_agg['type'] == 'download']
    ul = tile_agg[tile_agg['type'] == 'upload']

    dl_stats = dl.groupby(['year_q', 'province']).apply(lambda g: pd.Series({
        'avg_d_mbps': np.average(g['tile_mean'], weights=g['n']),
        'avg_lat_ms_wt': np.average(g['tile_lat'], weights=g['n']),
        'total_tests': g['n'].sum(),
        'n_tiles': g['tile_id'].nunique(),
    }), include_groups=False).reset_index()

    ul_stats = ul.groupby(['year_q', 'province']).apply(lambda g: pd.Series({
        'avg_u_mbps': np.average(g['tile_mean'], weights=g['n']),
    }), include_groups=False).reset_index()

    master = pd.merge(dl_stats, ul_stats, on=['year_q', 'province'], how='outer')
    master = master.rename(columns={'year_q': 'quarter'})
    master['year'] = master['quarter'].str.slice(0, 4).astype(int)
    master['quarter.1'] = master['quarter'].str.slice(6, 7).astype(int)

    master['is_reliable'] = (master['total_tests'] >= 100) & (master['n_tiles'] >= 5)
    print(f"[{network_type}] province x quarter rows: {len(master)} | reliable: {master['is_reliable'].sum()} ({master['is_reliable'].mean():.1%})")

    master = master.merge(
        ref[['province_en', 'region', 'internet_tier', 'pop_2024', 'gdp_per_capita_raw_2021',
             'density_per_km2', 'gdp_per_capita_usd_ppp_2021', 'gdp_per_capita_thb_2021']],
        left_on='province', right_on='province_en', how='left'
    ).drop(columns=['province_en'])

    missing_ref = master[master['region'].isna()]['province'].unique()
    if len(missing_ref):
        print(f"[{network_type}] WARNING — provinces with no reference match: {list(missing_ref)}")

    return master


EXPORT_COLS = ['province', 'quarter', 'year', 'quarter.1', 'avg_d_mbps', 'avg_u_mbps',
               'avg_lat_ms_wt', 'total_tests', 'n_tiles', 'is_reliable', 'region',
               'internet_tier', 'pop_2024', 'gdp_per_capita_raw_2021', 'density_per_km2',
               'gdp_per_capita_usd_ppp_2021', 'gdp_per_capita_thb_2021']

In [5]:
ref = pd.read_csv(VN_REF_CSV)

---
## Part 1 — Broadband

In [6]:
broadband_master = build_province_quarterly(tile_agg_all, 'broadband', ref)
broadband_master.head()

[broadband] tile x quarter x type rows: 5,568


[broadband] province x quarter rows: 733 | reliable: 210 (28.6%)


,quarter,province,avg_d_mbps,avg_lat_ms_wt,total_tests,n_tiles,avg_u_mbps,year,quarter.1,is_reliable,region,internet_tier,pop_2024,gdp_per_capita_raw_2021,density_per_km2,gdp_per_capita_usd_ppp_2021,gdp_per_capita_thb_2021
0,2023-Q1,An Giang,29.349034,112.389486,759.0,10.0,31.547373,2023,1,True,Mekong Delta,1,2057000,3791.46,540,10591.12,338704.14
1,2023-Q1,Bà Rịa–Vũng Tàu,35.333411,112.670980,1558.0,7.0,28.657948,2023,1,True,Southeast,1,1303000,36786.39,580,102759.68,3286254.54
2,2023-Q1,Bình Dương,31.278105,107.479716,747.0,11.0,25.175450,2023,1,True,Southeast,2,2564000,3663.54,901,10233.79,327276.61
3,2023-Q1,Bình Phước,29.091671,124.483435,278.0,8.0,24.916346,2023,1,True,Southeast,2,1313000,3606.56,145,10074.62,322186.39
4,2023-Q1,Bình Thuận,31.337100,97.125557,314.0,5.0,29.936516,2023,1,True,South Central Coast,2,1498000,3090.17,155,8632.13,276055.50


In [7]:
out_bb = broadband_master[EXPORT_COLS].copy()
OUT_PATH_BB = '../../data/exports/ndt7_vietnam_province_quarterly.csv'
out_bb.to_csv(OUT_PATH_BB, index=False)
print(f"Exported {len(out_bb)} rows -> {OUT_PATH_BB}")
out_bb.head(3)

Exported 733 rows -> ../../data/exports/ndt7_vietnam_province_quarterly.csv


,province,quarter,year,quarter.1,avg_d_mbps,avg_u_mbps,avg_lat_ms_wt,total_tests,n_tiles,is_reliable,region,internet_tier,pop_2024,gdp_per_capita_raw_2021,density_per_km2,gdp_per_capita_usd_ppp_2021,gdp_per_capita_thb_2021
0,An Giang,2023-Q1,2023,1,29.349034,31.547373,112.389486,759.0,10.0,True,Mekong Delta,1,2057000,3791.46,540,10591.12,338704.14
1,Bà Rịa–Vũng Tàu,2023-Q1,2023,1,35.333411,28.657948,112.670980,1558.0,7.0,True,Southeast,1,1303000,36786.39,580,102759.68,3286254.54
2,Bình Dương,2023-Q1,2023,1,31.278105,25.175450,107.479716,747.0,11.0,True,Southeast,2,2564000,3663.54,901,10233.79,327276.61


---
## Part 2 — Mobile/Cellular

In [8]:
mobile_master = build_province_quarterly(tile_agg_all, 'cellular', ref)
mobile_master.head()

[cellular] tile x quarter x type rows: 1,057


[cellular] province x quarter rows: 396 | reliable: 4 (1.0%)


,quarter,province,avg_d_mbps,avg_lat_ms_wt,total_tests,n_tiles,avg_u_mbps,year,quarter.1,is_reliable,region,internet_tier,pop_2024,gdp_per_capita_raw_2021,density_per_km2,gdp_per_capita_usd_ppp_2021,gdp_per_capita_thb_2021
0,2023-Q1,Bà Rịa–Vũng Tàu,22.548900,197.897375,16.0,2.0,14.552571,2023,1,False,Southeast,1,1303000,36786.39,580,102759.68,3286254.54
1,2023-Q1,Bình Dương,9.071973,146.487286,7.0,2.0,3.658284,2023,1,False,Southeast,2,2564000,3663.54,901,10233.79,327276.61
2,2023-Q1,Bình Phước,38.013270,141.964667,3.0,1.0,NaN,2023,1,False,Southeast,2,1313000,3606.56,145,10074.62,322186.39
3,2023-Q1,Bình Thuận,40.451301,115.547764,89.0,3.0,31.502123,2023,1,False,South Central Coast,2,1498000,3090.17,155,8632.13,276055.50
4,2023-Q1,Bình Định,11.630636,278.179689,103.0,2.0,8.800083,2023,1,False,South Central Coast,2,1679000,3089.10,245,8629.14,275959.91


In [9]:
out_mb = mobile_master[EXPORT_COLS].copy()
OUT_PATH_MB = '../../data/exports/ndt7_vietnam_mobile_province_quarterly.csv'
out_mb.to_csv(OUT_PATH_MB, index=False)
print(f"Exported {len(out_mb)} rows -> {OUT_PATH_MB}")
out_mb.head(3)

Exported 396 rows -> ../../data/exports/ndt7_vietnam_mobile_province_quarterly.csv


,province,quarter,year,quarter.1,avg_d_mbps,avg_u_mbps,avg_lat_ms_wt,total_tests,n_tiles,is_reliable,region,internet_tier,pop_2024,gdp_per_capita_raw_2021,density_per_km2,gdp_per_capita_usd_ppp_2021,gdp_per_capita_thb_2021
0,Bà Rịa–Vũng Tàu,2023-Q1,2023,1,22.548900,14.552571,197.897375,16.0,2.0,False,Southeast,1,1303000,36786.39,580,102759.68,3286254.54
1,Bình Dương,2023-Q1,2023,1,9.071973,3.658284,146.487286,7.0,2.0,False,Southeast,2,2564000,3663.54,901,10233.79,327276.61
2,Bình Phước,2023-Q1,2023,1,38.013270,NaN,141.964667,3.0,1.0,False,Southeast,2,1313000,3606.56,145,10074.62,322186.39


## Summary

- Input: 24.7M raw NDT7 test records for Vietnam (2023–2025): 22.0M broadband, 1.27M
  cellular, 1.39M hosting (excluded)
- Output: province x quarter aggregates for Broadband and Mobile separately, tile-binned at
  Ookla's zoom-16 resolution, same `is_reliable` threshold as every Ookla country notebook
- Processed in streaming batches (memory-safe) — Broadband numbers should match the original
  single-part prep notebook exactly